In [0]:
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/Volumes/dbr_dev_ua5816bd/viktoriia_kalenichenko/raw_data/europe_cereal_yield_climate_1990_2022.csv")

df = df.filter(df["AREA"] != "Russian Federation")

In [0]:
display(df)

In [0]:
df.printSchema()

In [0]:
df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dbr_dev_ua5816bd.viktoriia_kalenichenko.cereal_climate")

In [0]:
#select operation + craetion tables
cereals_df = df.select(
    "AREA",
    "ITEM",
    "YEAR",
    "AREA_HARVESTED",
    "PRODUCTION_QUANTITY",
    "YIELD"
)

cereals_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dbr_dev_ua5816bd.viktoriia_kalenichenko.cereals")

In [0]:
climate_df = df.select(
    "AREA",
    "YEAR",
    "WB_CCKP_CDD",
    "WB_CCKP_CDD65",
    "WB_CCKP_CSDI",
    "WB_CCKP_CWD",
    "WB_CCKP_FD",
    "WB_CCKP_HD30",
    "WB_CCKP_HD35",
    "WB_CCKP_HD40",
    "WB_CCKP_HD42",
    "WB_CCKP_HD45",
    "WB_CCKP_HD50",
    "WB_CCKP_HDD65",
    "WB_CCKP_HI35",
    "WB_CCKP_HI37",
    "WB_CCKP_HI39",
    "WB_CCKP_HI41",
    "WB_CCKP_HURS",
    "WB_CCKP_ID",
    "WB_CCKP_PR",
    "WB_CCKP_R20MM",
    "WB_CCKP_R50MM",
    "WB_CCKP_R95PTOT",
    "WB_CCKP_RX1DAY",
    "WB_CCKP_RX5DAY",
    "WB_CCKP_SD",
    "WB_CCKP_TAS",
    "WB_CCKP_TASMAX",
    "WB_CCKP_TASMIN",
    "WB_CCKP_TNN",
    "WB_CCKP_TR",
    "WB_CCKP_TR23",
    "WB_CCKP_TR26",
    "WB_CCKP_TR29",
    "WB_CCKP_TR32",
    "WB_CCKP_TX84RR",
    "WB_CCKP_TXX",
    "WB_CCKP_WSDI"
)

climate_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dbr_dev_ua5816bd.viktoriia_kalenichenko.climate")

In [0]:
climate_factors_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/Volumes/dbr_dev_ua5816bd/viktoriia_kalenichenko/raw_data/climatic_factors.csv")

climate_factors_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dbr_dev_ua5816bd.viktoriia_kalenichenko.climatic_factors")

display(climate_factors_df)

In [0]:
environment = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/Volumes/dbr_dev_ua5816bd/viktoriia_kalenichenko/raw_data/environment_clean.csv")

environment.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dbr_dev_ua5816bd.viktoriia_kalenichenko.environment")

display(environment)

In [0]:
#filter operation
display(df.filter(df["YEAR"] >= 2020))

In [0]:
#groupBy operation
display(df.groupBy("AREA").avg("YIELD"))

In [0]:
joined_df = cereals_df \
    .join(
        climate_df,
        on=["AREA", "YEAR"],
        how="inner"
    ) \
    .join(
        environment,
        on=["AREA", "YEAR"],
        how="left"
    )

display(joined_df)

In [0]:
display(df.select("AREA").distinct().orderBy("AREA"))#other operation + showing which countries we analyze
display(df.select("AREA").distinct().orderBy("AREA"))

In [0]:
#checking for dublicates 

joined_df.groupBy("AREA", "YEAR") \
    .count() \
    .filter("count > 1") \
    .display()

In [0]:
joined_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "dbr_dev_ua5816bd.viktoriia_kalenichenko.joined_data"
    )

In [0]:
%sql
-- queries for the dashboard
SELECT
    AREA,
    ITEM,
    YEAR,
    YIELD
FROM dbr_dev_ua5816bd.viktoriia_kalenichenko.cereals
WHERE YEAR BETWEEN 2020 AND 2022
ORDER BY AREA, ITEM, YEAR;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- average cereal yield by country for 2020–2022

SELECT
    AREA,
    country_code,
    AVG(YIELD) AS avg_yield
FROM dbr_dev_ua5816bd.viktoriia_kalenichenko.joined_data
WHERE YEAR BETWEEN 2020 AND 2022
GROUP BY AREA, country_code
ORDER BY avg_yield DESC


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT
    AREA,
    YEAR,
    AVG(WB_CCKP_TAS) AS avg_temperature,
    AVG(YIELD) AS avg_yield
FROM  dbr_dev_ua5816bd.viktoriia_kalenichenko.cereal_climate
GROUP BY AREA, YEAR
ORDER BY AREA, YEAR;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql

SELECT
    AREA,
    YEAR,
    ITEM,
    AVG(WB_CCKP_TAS) AS avg_temperature,
    AVG(YIELD) AS avg_yield
FROM dbr_dev_ua5816bd.viktoriia_kalenichenko.cereal_climate
GROUP BY AREA, YEAR, ITEM
ORDER BY AREA, YEAR, ITEM;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql

SELECT
    AREA,
    YEAR,
    ITEM,
    AVG(WB_CCKP_PR) AS avg_precipitation,
    AVG(YIELD) AS avg_yield
FROM dbr_dev_ua5816bd.viktoriia_kalenichenko.cereal_climate
GROUP BY AREA, YEAR, ITEM
ORDER BY AREA, YEAR, ITEM;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT
    AREA,
    YEAR,
    AVG(WB_CCKP_PR) AS avg_precipitation,
    AVG(YIELD) AS avg_yield
FROM dbr_dev_ua5816bd.viktoriia_kalenichenko.cereal_climate
GROUP BY AREA, YEAR
ORDER BY AREA, YEAR;

In [0]:
%sql

SELECT
    AREA,
    AVG(freshwater_per_capita) AS avg_freshwater_per_capita,
    AVG(freshwater_withdrawals) AS avg_freshwater_withdrawals,
    AVG(YIELD) AS avg_yield
FROM dbr_dev_ua5816bd.viktoriia_kalenichenko.joined_data
GROUP BY AREA
ORDER BY avg_yield DESC;

In [0]:
country_codes = [
    row["country_code"]
    for row in environment
        .select("country_code")
        .distinct()
        .collect()
]

country_codes = [code for code in country_codes if code]

print(country_codes)

countries = ";".join(country_codes)

print(countries)

In [0]:
#external API
import requests

url = "https://api.worldbank.org/v2/country/LVA;POL;FRA;ITA;UKR;HRV;GBR;MLT;BLR;SVK;HUN;NOR;FIN;ALB;BIH;NLD;LUX;MNE;AUT;PRT;LTU;ROU;DNK;ESP;EST;IRL;SWE;SVN;GRC;BEL;MKD;DEU;MDA;BGR;SRB;CHE;CZE;ISL/indicator/NV.AGR.TOTL.ZS"

response = requests.get(
    url,
    params={
        "format": "json",
        "date": "1990:2022"
    }
)

data = response.json()

print(data)

In [0]:
import pandas as pd

rows = data[1]

api_df = pd.DataFrame([
    {
        "AREA": row["country"]["value"],
        "country_code": row["countryiso3code"],
        "YEAR": int(row["date"]),
        "agriculture_value_added_gdp": row["value"]
    }
    for row in rows
    if row["value"] is not None
])

display(api_df)

In [0]:
agriculture_gdp = spark.createDataFrame(api_df)

display(agriculture_gdp)

agriculture_gdp.printSchema()

In [0]:
agriculture_gdp.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "dbr_dev_ua5816bd.viktoriia_kalenichenko.agriculture_gdp"
    )

Optional Addition: Delta Lake is a data storage format that provides ACID transactions and schema enforcement, and supports Time Travel for viewing previous versions of data.